# Session 1: from a catalogue to a defensible result
**120 minutes, including a 10-minute break.**

You will inspect shapes, combine masks without losing row alignment, retain units,
read a FITS table and make a plot that exposes a selection effect.

The 400-row catalogue is entirely synthetic. Distances are assumed exact and
independent of the velocities; the velocities follow an injected linear relation.
This simplified exercise is not a cosmology analysis.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable
# The Hub pulls the course into your workspace before starting this notebook.
# Prefer that editable checkout; the local editable install is a fallback.
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / "jupyter_data_languages"]
for _candidate in _candidates:
    if (_candidate / "course2026/__init__.py").is_file() and (_candidate / "data/mock_galaxies.ecsv").is_file():
        sys.path.insert(0, str(_candidate))
        break
import course2026
ROOT = Path(course2026.__file__).resolve().parents[1]
plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11})

## 0–10 min: baseline check
Predict what the shape and first three rows will show. What are the units, which
columns carry measurement errors, and what metadata would you record with a real survey?

In [ ]:
table = QTable.read(ROOT / 'data/mock_galaxies.ecsv')
print('Rows:', len(table), '| synthetic:', table.meta['synthetic'])
print(table[:3])
print('Distance unit:', table['distance'].unit)

## 10–40 min: arrays, shapes and masks
Never apply different row filters separately to distance, velocity and errors.
Create one common mask, report what it removes, and apply it to every column.
The apparent-magnitude column is a plain numeric logarithmic magnitude scale;
smaller numbers mean brighter objects. The other physical columns retain units.

In [ ]:
distance = table['distance'].to_value(u.Mpc)
velocity = table['velocity'].to_value(u.km / u.s)
sigma = table['velocity_error'].to_value(u.km / u.s)
magnitude = np.asarray(table['apparent_magnitude'])
valid = (np.isfinite(distance) & np.isfinite(velocity) & np.isfinite(sigma)
         & np.isfinite(magnitude) & (distance > 0) & (sigma > 0))
print('Shape:', distance.shape, '| usable:', valid.sum(), '| removed:', (~valid).sum())
assert valid.sum() == 395
assert distance[valid].shape == velocity[valid].shape == sigma[valid].shape

### Task A — ten minutes
Count missing velocities and nonpositive error bars separately. Which objects
are excluded, and why should they not simply be assigned a zero velocity?
Change the magnitude cut below to 15.5 and 17.5. Predict the count before running.

**Hint:** `np.count_nonzero(condition)` counts true values; `table[mask]` selects whole rows.

In [ ]:
magnitude_limit = 16.5
selected = valid & (magnitude < magnitude_limit)
print('Selected:', selected.sum(), 'of', valid.sum(), 'usable rows')
assert 0 < selected.sum() < valid.sum()
# Many observations, three features: one row scale must have shape (N, 1).
features = np.column_stack([distance[valid], velocity[valid], sigma[valid]])
scale_per_object = np.ones(features.shape[0])
print('Feature matrix:', features.shape, '| row scale:', scale_per_object[:, None].shape)
assert (features * scale_per_object[:, None]).shape == features.shape

## 40–60 min: units and FITS
Do not infer a unit from the size of a number. Use the data definition or header.
Astropy converts compatible units and rejects incompatible ones.
A km/s ↔ m/s mistake is a factor of 1000 even when a plot still looks plausible.

In [ ]:
velocity_ms = table['velocity'].to(u.m / u.s)
np.testing.assert_allclose(velocity_ms.value[valid], 1000 * velocity[valid])
try:
    table['distance'].to(u.s)
except u.UnitConversionError:
    print('Expected: a distance cannot be converted to seconds')

In [ ]:
# A local round-trip teaches FITS I/O without a survey download.
# Inspect metadata separately: ECSV supports richer arbitrary metadata than FITS.
import tempfile
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / 'mock_subset.fits'
    subset = table[['distance', 'velocity', 'velocity_error']][valid].copy()
    subset.meta.clear()
    subset.write(path)
    restored = QTable.read(path)
    np.testing.assert_allclose(restored['distance'].to_value(u.Mpc), distance[valid])
    print('FITS round-trip rows:', len(restored), '| unit:', restored['distance'].unit)

### Task B — five minutes
Read the FITS round-trip code aloud to a partner. Which information did we
explicitly omit? Why is `to_value(u.Mpc)` safer than taking `.value` without
checking the stored unit? Identify one test you would add when reading real data.

**60–70 min: break.**

## 70–95 min: plot the data and its uncertainty
Use labelled physical axes, visible uncertainty and a caption that identifies the
selection. Avoid showing 400 error bars when a readable subset will do.
The errors here are known independent Gaussian one-sigma errors in velocity only.

In [ ]:
fig, ax = plt.subplots()
indices = np.flatnonzero(valid)[::8]
ax.errorbar(distance[indices], velocity[indices], yerr=sigma[indices],
            fmt='o', ms=3, alpha=0.7, capsize=2, label='Every eighth usable mock object')
ax.set(xlabel='Independent mock distance [Mpc]', ylabel='Velocity [km/s]',
       title='Synthetic catalogue: measurement uncertainty')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 95–115 min: investigate a survey selection
Use the **same bin edges** for both populations. Counts show which objects are
lost; a density histogram normalises the area, so it answers a different question.

In [ ]:
bins = np.linspace(20, 200, 13)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, density in zip(axes, [False, True]):
    ax.hist(distance[valid], bins=bins, histtype='step', lw=2, density=density,
            label='All usable mock objects')
    ax.hist(distance[selected], bins=bins, histtype='step', lw=2, density=density,
            label=f'Apparent magnitude < {magnitude_limit}')
    ax.set(xlabel='Distance [Mpc]', ylabel='Probability density [1/Mpc]' if density else 'Objects')
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()
print('Mean distances [Mpc]:', distance[valid].mean().round(1), distance[selected].mean().round(1))

### Task C — ten minutes
Explain why the selected catalogue has a different distance distribution. Write a
three-sentence figure caption: what is plotted, what was removed, and which
inference the plot does **not** justify. Would a velocity fit necessarily become
biased by this cut? Identify the model assumptions you would need to answer.

**Hint:** in this mock, selection depends on distance and magnitude, not velocity
noise. A change in a sample distribution is not automatically a bias in every estimator.

## 115–120 min: exit check
Explain why there are 395 usable rows, how broadcasting works for `(N, 3)` and
`(N, 1)`, why units must come from provenance, and why selected counts differ.
Save the notebook and verify it with Restart and Run All. A plausible plot is
one check, not proof of a scientifically valid inference.